In [ ]:
import json
import tensorflow as tf

from keras.layers import (
    Input, Dense, Embedding, LSTM, Dropout,
    Add, TimeDistributed, RepeatVector
)
from keras.models import Model


def define_model(vocab_size, max_length):


    # image features
    image_features_input = Input(shape=(512,), name="image_features")

    image_features = Dropout(0.5, name="image_dropout")(image_features_input)

    image_features = Dense(256, activation="relu", name="image_dense")(image_features)

    # lstm
    caption_input = Input(shape=(max_length,), name="caption")

    caption_embedding = Embedding(vocab_size, 256, mask_zero=True, name="caption_embedding")(caption_input)
    
    caption_embedding = Dropout(0.5, name="caption_dropout")(caption_embedding)

    hidden_state_sequence = LSTM(256, return_sequences=True, name="lstm")(caption_embedding)

    # image conditioning
    image_features_sequence = RepeatVector(max_length, name="repeat_vector")(image_features)

    # Merge + classifier (Add è diverso da Cocatenate, Add somma i tensori elemento per elemento, Concatenate li concatena lungo un asse)
    merged_representation = Add()([hidden_state_sequence, image_features_sequence])
    
    merged_representation = TimeDistributed(Dense(256, activation="relu"), name="classifier_dense")(merged_representation)
    
    vocab_distribution_per_timestep = TimeDistributed(Dense(vocab_size, activation="softmax"), name="classifier")(merged_representation)

    # model
    model = Model(
        inputs=[image_features_input, caption_input],
        outputs=vocab_distribution_per_timestep,
        name="image_caption_generator"
    )
    
    # summarize model
    print(model.summary())

    return model


# =========================================================
# LOAD CONFIG
# =========================================================
with open("../config/vec_config.json", "r") as f:
    vec_config = json.load(f)

max_length = vec_config["output_sequence_length"]
max_length -= 1  # teacher forcing shift

vocab_size = vec_config["vocabulary_size"]

# =========================================================
# BUILD MODEL
# =========================================================
model = define_model(
    vocab_size,
    max_length,
)


2026-06-28 18:43:27.929241: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model: "image_caption_generator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ caption             │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image_features      │ (None, 512)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ caption_embedding   │ (None, 16, 256)   │  1,939,968 │ caption[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image_dropout       │ (None, 512)       │          0 │ image_features[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ caption_dropout     │ (None, 16, 256)   │          0 │ caption_embeddin… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 16)        │          0 │ caption[0][0]     │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image_dense (Dense) │ (None, 256)       │    131,328 │ image_dropout[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 16, 256)   │    525,312 │ caption_dropout[… │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector       │ (None, 16, 256)   │          0 │ image_dense[0][0] │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 16, 256)   │          0 │ lstm[0][0],       │
│                     │                   │            │ repeat_vector[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier_dense    │ (None, 16, 256)   │     65,792 │ add[0][0]         │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier          │ (None, 16, 7578)  │  1,947,546 │ classifier_dense… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,609,946 (17.59 MB)

 Trainable params: 4,609,946 (17.59 MB)

 Non-trainable params: 0 (0.00 B)

None


In [2]:
import tensorflow as tf
import numpy as np


# =========================================================
# LOAD PRECOMPUTED CNN FEATURES
# =========================================================

features = np.load("../data/images/features.npz")


# =========================================================
# TFRECORD PARSING (UNCHANGED STRUCTURE)
# =========================================================

def parse_example(proto):
    feature_desc = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "input": tf.io.FixedLenFeature([16], tf.int64),
        "target": tf.io.FixedLenFeature([16], tf.int64),
    }

    return tf.io.parse_single_example(proto, feature_desc)


# =========================================================
# SAFE FEATURE LOOKUP
# =========================================================

def load_features(example):

    image_id = example["image"]

    feature = tf.py_function(
        func=lambda x: features[x.numpy().decode()].astype(np.float32),
        inp=[image_id],
        Tout=tf.float32,
    )

    feature.set_shape((512,))  # type: ignore
    
    example["image"] = feature
    return example


# =========================================================
# PREPARE MODEL INPUT/OUTPUT
# =========================================================

def prepare(example):

    return {
        "image_features": example["image"],
        "caption": example["input"]
    }, example["target"]


# =========================================================
# DATASET PIPELINE
# =========================================================

BATCH_SIZE = 64


def build_dataset_from_tf_record(tfrecord_path, batch_size=BATCH_SIZE, dataset_size=None):
    dataset = tf.data.TFRecordDataset(tfrecord_path)

    dataset = dataset.map(parse_example, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(load_features, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(prepare, num_parallel_calls=tf.data.AUTOTUNE)

    if dataset_size is not None:
        dataset = dataset.shuffle(dataset_size, reshuffle_each_iteration=True)

    dataset = dataset.repeat()
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

train_set = build_dataset_from_tf_record(
    "../data/records/train.tfrecord",
    dataset_size=30000,
    batch_size=BATCH_SIZE
)

val_set = build_dataset_from_tf_record(
    "../data/records/val.tfrecord",
    batch_size=BATCH_SIZE
)



In [ ]:
import math
import keras


@keras.saving.register_keras_serializable()
def masked_loss(y_true, y_pred):
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    loss = keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    loss *= mask # type: ignore
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)


model.compile(loss=masked_loss, optimizer="adam")

NUM_TRAIN_STEPS = math.ceil(30000 / BATCH_SIZE)
NUM_VAL_STEPS = math.ceil(5000 / BATCH_SIZE)

model.fit(
    train_set,
    epochs=30,
    steps_per_epoch=NUM_TRAIN_STEPS,
    validation_data=val_set,
    validation_steps=NUM_VAL_STEPS
)

model.save("../output/icg.keras")

l = keras.models.load_model("../output/icg.keras", custom_objects={"masked_loss": masked_loss})



Epoch 1/30


/Users/cocco/Desktop/icg/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-06-28 18:43:43.812436: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] fused(ShuffleDatasetV3:5,RepeatDataset:6): Filling up shuffle buffer (this may take a while): 16552 of 30000
2026-06-28 18:43:51.986964: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step - loss: 5.0807